In [ ]:
# Import required libraries
import os
import re
import pandas as pd
from openai import OpenAI
import json
from collections import Counter
import time
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Set default model
MODEL = "gpt-4o-mini"

In [3]:
def call_openai(prompt, 
                system_message="You are a helpful assistant.", 
                temperature=0.7, 
                max_tokens=None, 
                seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }
        
        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def display_response(title, response, params=None):
    """Display a response with formatting"""
    print(f"\n{'='*60}")
    print(f"📝 {title}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")

def count_tokens_approx(text):
    """Approximate token count (roughly 4 characters per token)"""
    return len(text) // 4

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


In [3]:
# Initial simple prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."
"""
 
# Test it once
result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

Sentiment Analysis Result:
This customer message can be classified as **Positive Feedback**. The customer expresses initial concerns but ultimately shares a positive experience after using the product.


In [4]:
# Initial simple prompt for product description
product_prompt_v1 = """
Create a product description for a whey protein powder that costs $69.99.
"""
 
# Test it once
result = call_openai(product_prompt_v1)
print("Product Description Result:")
print(result)

Product Description Result:
**Product Name: PurePower Whey Protein Powder**

**Price: $69.99**

Elevate your fitness journey with **PurePower Whey Protein Powder**, the ultimate supplement designed for athletes, fitness enthusiasts, and anyone seeking to enhance their protein intake. Priced at $69.99, this high-quality protein powder offers exceptional value for those who demand the best from their nutrition.

**Key Features:**

- **Premium Quality Protein**: Each serving delivers 25 grams of pure, high-quality whey protein isolate, sourced from grass-fed cows for superior taste and bioavailability. Our formula is designed to support muscle recovery, growth, and overall performance.

- **Low in Sugar and Fat**: With only 1 gram of sugar and 1.5 grams of fat per serving, PurePower Whey is perfect for those following a low-carb or calorie-conscious diet. Enjoy the benefits of protein without the added sugars and calories.

- **Delicious Flavors**: Available in a variety of mouthwatering 

In [5]:
# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."
"""
 
# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Data Extraction Result:
Here is the extracted information from the customer feedback:

- **Item Ordered**: #43
- **Order Date**: May 4th
- **Delivery Experience**: Took long
- **Packaging Condition**: Not damaged


In [6]:
# Run the same prompt 5 times with temperature=0.7
prompt = sentiment_prompt_v1
temperature = 0.7
num_runs = 5

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
sentiment_responses_5 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 5 times with temperature=0.7
Prompt: '
Classify this customer message: "I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."
'


🔄 Run #1:
The customer message can be classified as **Positive Feedback**. The customer initially had concerns but is now satisfied with the product after using it for a week.
--------------------------------------------------------------------------------

🔄 Run #2:
The customer message can be classified as "Positive Feedback." The customer expresses initial concerns but ultimately ends up pleasantly surprised after using the product, indicating satisfaction.
--------------------------------------------------------------------------------

🔄 Run #3:
The customer message can be classified as **Positive Feedback**. The customer expresses initial concerns but ultimately finds the product satisfactory after using it for a week.
-------------------------------------------------------

In [7]:
# Run the same prompt 5 times with temperature=0.7
prompt = product_prompt_v1
temperature = 0.7
num_runs = 5

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
product_responses_5 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 5 times with temperature=0.7
Prompt: '
Create a product description for a whey protein powder that costs $69.99.
'


🔄 Run #1:
**Product Name: PurePro Whey Protein Powder**

**Price: $69.99**

**Description:**

Unlock your fitness potential with PurePro Whey Protein Powder, the ultimate supplement designed for athletes, fitness enthusiasts, and anyone seeking to elevate their nutritional game. Priced at just $69.99, this premium whey protein powder is crafted to deliver unparalleled quality, taste, and results.

**Key Features:**

- **Superior Quality Whey:** Sourced from grass-fed cows, our whey protein undergoes minimal processing to ensure you receive a product that is rich in essential amino acids and free from harmful additives. Each serving provides 25g of high-quality protein to support muscle growth and recovery.

- **Delicious Flavors:** Available in a variety of mouthwatering flavors, including Rich Chocolate, Creamy Vanilla, and Strawberry Bliss, Pure

In [8]:
# Run the same prompt 5 times with temperature=0.7
prompt = extraction_prompt_v1
temperature = 0.7
num_runs = 5

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
extraction_responses_5 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 5 times with temperature=0.7
Prompt: '
Extract information from this customer feedback: "I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."
'


🔄 Run #1:
Here is the extracted information from the customer feedback:

- **Item Ordered**: #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
--------------------------------------------------------------------------------

🔄 Run #2:
Here is the extracted information from the customer feedback:

- **Item Number**: #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
--------------------------------------------------------------------------------

🔄 Run #3:
Here is the extracted information from the customer feedback:

- **Item Ordered**: #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
---

In [9]:
# Run the same prompt 10 times with temperature=0.7
prompt = sentiment_prompt_v1
temperature = 0.7
num_runs = 10

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
sentiment_responses_10 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 10 times with temperature=0.7
Prompt: '
Classify this customer message: "I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."
'


🔄 Run #1:
The customer message can be classified as "Positive Feedback." The customer initially had concerns but has found the product to be satisfactory after a week of use.
--------------------------------------------------------------------------------

🔄 Run #2:
The customer message can be classified as **positive feedback**. The customer initially had concerns, but their experience after using the product has led to a pleasant surprise, indicating satisfaction with the product.
--------------------------------------------------------------------------------

🔄 Run #3:
The customer message can be classified as **Positive Feedback**. The customer expresses initial concerns but ultimately conveys satisfaction with the product after a week of use.
------------------------------

In [10]:
# Run the same prompt 10 times with temperature=0.7
prompt = product_prompt_v1
temperature = 0.7
num_runs = 10

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
product_responses_10 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 10 times with temperature=0.7
Prompt: '
Create a product description for a whey protein powder that costs $69.99.
'


🔄 Run #1:
**Product Name: PurePower Whey Protein Powder**

**Price: $69.99**

**Product Description:**

Unlock your fitness potential with PurePower Whey Protein Powder, the ultimate supplement designed for those who demand excellence in their nutritional regimen. Priced at $69.99, this premium protein powder is crafted from high-quality whey protein isolate, offering you a potent source of protein to fuel your workouts and support muscle recovery.

**Key Features:**

- **High-Quality Protein Source:** Each serving delivers 25g of pure whey protein isolate, ensuring you get the essential amino acids your body craves for optimal muscle repair and growth.
- **Low in Carbs and Fat:** With only 2g of carbs and 1g of fat per serving, PurePower Whey is the perfect choice for those looking to maintain a lean physique without sacrificing taste or nutriti

In [11]:
# Run the same prompt 10 times with temperature=0.7
prompt = extraction_prompt_v1
temperature = 0.7
num_runs = 10

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
extraction_responses_10 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 10 times with temperature=0.7
Prompt: '
Extract information from this customer feedback: "I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."
'


🔄 Run #1:
Here is the extracted information from the customer feedback:

- **Item Ordered**: Item #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
--------------------------------------------------------------------------------

🔄 Run #2:
Here are the key pieces of information extracted from the customer feedback:

- **Item Ordered**: Item #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
--------------------------------------------------------------------------------

🔄 Run #3:
Here is the extracted information from the customer feedback:

- **Item Ordered**: #43
- **Order Date**: May 4th
- **Delivery Time**: Took long
- **Packaging Condition**: N

### Sentiment Analysis – Observations

The initial test provided a first result, while the 5-run and 10-run tests showed consistent sentiment classification. All responses correctly identified the feedback as positive, although wording, capitalization, and formatting varied. The 10-run test also introduced an alternative classification term ("Satisfaction"), highlighting the lack of standardized output labels.

### Product Description – Observations

The initial test already generated a very detailed description with several product attributes that were not provided in the prompt. The 5-run test revealed variations in product names, nutritional information, and formatting. The 10-run test confirmed these inconsistencies and introduced additional differences, such as 25g versus 30g of protein per serving. The price remained consistent throughout all tests.

### Data Extraction – Observations

The initial test successfully extracted all relevant information from the customer feedback. The 5-run test maintained accurate results but showed minor differences in field names and formatting. The 10-run test revealed more formatting variations, including different labels and representations of the item ID. However, all extracted information remained correct across the tests.

In [12]:
# Run the same prompt 15 times with temperature=0.7
prompt = sentiment_prompt_v1
temperature = 0.7
num_runs = 15

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
sentiment_responses_15 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 15 times with temperature=0.7
Prompt: '
Classify this customer message: "I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."
'


🔄 Run #1:
The customer message can be classified as **positive feedback**. The customer expresses initial concerns but ultimately has had a positive experience after using the product for a week.
--------------------------------------------------------------------------------

🔄 Run #2:
The customer message can be classified as **positive feedback**. The customer expresses initial concerns but indicates that their experience has improved, leading to a pleasant surprise.
--------------------------------------------------------------------------------

🔄 Run #3:
The customer message can be classified as **positive feedback**. The customer expresses initial concerns but ultimately shares a positive experience after using the product for a week.
-------------------------------------

In [13]:
# Run the same prompt 15 times with temperature=0.7
prompt = product_prompt_v1
temperature = 0.7
num_runs = 15

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
product_responses_15 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 15 times with temperature=0.7
Prompt: '
Create a product description for a whey protein powder that costs $69.99.
'


🔄 Run #1:
**Product Description: Premium Whey Protein Powder - 2 lbs**

Elevate your fitness journey with our Premium Whey Protein Powder, expertly crafted for athletes, fitness enthusiasts, and anyone looking to boost their protein intake. Priced at $69.99, this top-tier supplement is more than just a protein source; it’s your partner in achieving peak performance and optimal recovery.

**Key Features:**

- **High-Quality Protein Blend:** Each serving packs a powerful 25 grams of pure whey protein derived from grass-fed cows, ensuring you receive the essential amino acids your body craves for muscle growth and repair.

- **Delicious Flavors:** Available in a variety of mouthwatering flavors, including Rich Chocolate, Creamy Vanilla, and Strawberry Bliss, our whey protein powder makes it easy to enjoy your post-workout shake or morning smoothie.


In [14]:
# Run the same prompt 15 times with temperature=0.7
prompt = extraction_prompt_v1
temperature = 0.7
num_runs = 15

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{prompt}'\n")
print("="*80)

responses = []
for i in range(num_runs):
    response = call_openai(prompt, temperature=temperature)
    responses.append(response)
    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)
    time.sleep(0.5)  # Small delay to avoid rate limiting
    
extraction_responses_15 = responses.copy()

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(responses))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in responses) / len(responses):.0f} characters")

Running the same prompt 15 times with temperature=0.7
Prompt: '
Extract information from this customer feedback: "I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."
'


🔄 Run #1:
Here is the extracted information from the customer feedback:

- **Order Item**: #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
--------------------------------------------------------------------------------

🔄 Run #2:
Here is the extracted information from the customer feedback:

- **Order Number**: #43
- **Order Date**: May 4th
- **Delivery Experience**: Took a long time
- **Packaging Condition**: Not damaged
--------------------------------------------------------------------------------

🔄 Run #3:
Here is the extracted information from the customer feedback:

- Item ordered: #43
- Order date: May 4th
- Delivery experience: Took long
- Packaging condition: Not damaged
--------------------------

In [ ]:
# Collect all saved test results
test_results = {
    "Sentiment": {
        5: sentiment_responses_5,
        10: sentiment_responses_10,
        15: sentiment_responses_15
    },
    "Product": {
        5: product_responses_5,
        10: product_responses_10,
        15: product_responses_15
    },
    "Extraction": {
        5: extraction_responses_5,
        10: extraction_responses_10,
        15: extraction_responses_15
    }
}

# Analyze results
analysis = []

for task, runs in test_results.items():
    for num_runs, responses in runs.items():

        unique = len(set(responses))
        avg_length = sum(len(r) for r in responses) / len(responses)

        analysis.append({
            "Task": task,
            "Runs": num_runs,
            "Unique Responses": unique,
            "Unique Rate (%)": round(unique / num_runs * 100, 1),
            "Avg. Length": round(avg_length)
        })

# Create analysis table
analysis_df = pd.DataFrame(analysis)

display(analysis_df)

,Task,Runs,Unique Responses,Unique Rate (%),Avg. Length
0,Sentiment,5,5,100.0,183
1,Sentiment,10,10,100.0,181
2,Sentiment,15,15,100.0,174
3,Product,5,5,100.0,2270
4,Product,10,10,100.0,2299
5,Product,15,15,100.0,2355
6,Extraction,5,4,80.0,197
7,Extraction,10,10,100.0,194
8,Extraction,15,11,73.3,194


In [ ]:
# Define task-specific evaluation rules
evaluation_rules = {
    "Sentiment": {
        "Correct positive classification": lambda r:
            bool(re.search(r"\bpositive feedback\b", r, re.I)),

        "Bold classification formatting": lambda r:
            bool(re.search(r"\*\*positive feedback\*\*", r, re.I))
    },

    "Product": {
        "Correct price ($69.99)": lambda r:
            "$69.99" in r,

        "Unprovided nutritional values": lambda r:
            bool(re.search(r"\b\d+(\.\d+)?\s*(g|grams?|calories)\b", r, re.I)),

        "Guarantee or return policy claims": lambda r:
            bool(re.search(r"guarantee|return policy|money.back", r, re.I)),

        "Testing or certification claims": lambda r:
            bool(re.search(r"certified|third.party tested|rigorously tested", r, re.I))
    },

    "Extraction": {
        "Item ID present": lambda r:
            bool(re.search(r"#?43\b", r)),

        "Order date present": lambda r:
            "may 4th" in r.lower(),

        "Delayed delivery recognized": lambda r:
            bool(re.search(r"took (a )?long", r, re.I)),

        "Packaging condition correct": lambda r:
            "not damaged" in r.lower(),

        "Incorrect Order Number label": lambda r:
            bool(re.search(r"order number", r, re.I))
    }
}

# Evaluate stored responses
failure_analysis = []

for task, runs in test_results.items():
    for num_runs, responses in runs.items():
        for pattern, check in evaluation_rules[task].items():

            matching_runs = [
                i for i, response in enumerate(responses, 1)
                if check(response)
            ]

            failure_analysis.append({
                "Task": task,
                "Runs": num_runs,
                "Pattern": pattern,
                "Occurrences": len(matching_runs),
                "Rate (%)": round(len(matching_runs) / num_runs * 100, 1),
                "Example Run": matching_runs[0] if matching_runs else None
            })

failure_df = pd.DataFrame(failure_analysis)

# Show 15-run results
display(failure_df[failure_df["Runs"] == 15])

,Task,Runs,Pattern,Occurrences,Rate (%),Example Run
4,Sentiment,15,Correct positive classification,15,100.0,1.0
5,Sentiment,15,Bold classification formatting,10,66.7,1.0
14,Product,15,Correct price ($69.99),15,100.0,1.0
15,Product,15,Unprovided nutritional values,15,100.0,1.0
16,Product,15,Guarantee or return policy claims,2,13.3,1.0
17,Product,15,Testing or certification claims,7,46.7,2.0
28,Extraction,15,Item ID present,15,100.0,1.0
29,Extraction,15,Order date present,15,100.0,1.0
30,Extraction,15,Delayed delivery recognized,15,100.0,1.0
31,Extraction,15,Packaging condition correct,15,100.0,1.0


## Step 5 – Failure Analysis Summary

### Sentiment Analysis

The sentiment analysis prompt showed high content consistency across all tests. In the 15-run test, all responses correctly classified the customer feedback as positive (100%). However, all 15 responses were unique, with variations in wording, capitalization, and formatting. Only 66.7% used bold formatting for the classification. This shows that the prompt reliably identifies the sentiment of the given message but does not produce a standardized output.

### Product Description Generation

The product description prompt showed the most significant weaknesses. While all 15 responses included the correct price of $69.99 (100%), every response introduced nutritional values that were not provided in the original prompt. Additionally, our automated analysis detected testing or certification claims in 46.7% of responses and guarantee or return policy claims in 13.3%. Product names, ingredients, nutritional values, and descriptions also varied between runs. These results highlight the risk of generating unsupported product information and demonstrate the need for clearer instructions and constraints.

### Data Extraction

The data extraction prompt successfully identified all four key information values in every run (100%). However, only 11 out of 15 responses were unique, and field names and formatting varied considerably. In one response (6.7%), the item ID was incorrectly labeled as an "Order Number", demonstrating a potential semantic extraction error. While the extracted information was generally accurate, the lack of a standardized output structure could create problems for automated data processing.

### Overall Findings

The tests demonstrated that **content accuracy and output consistency are not necessarily the same**. Sentiment analysis and data extraction performed well in identifying the required information, while product description generation frequently introduced unsupported details. Increasing the number of runs helped reveal additional failure patterns, particularly in formatting, field naming, and generated product attributes. The automated checks provided measurable indicators, although they do not capture every possible error. Since each task was tested with one fixed input, the results also do not establish reliability across different customer messages or products.

In [4]:
sentiment_prompt_v2 = """
Classify this customer message:
"I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."

Choose exactly one of the following classifications:
- Positive
- Negative
- Neutral

Respond with only the selected classification as a single word.
Do not include explanations, additional text, or formatting.
"""

result = call_openai(sentiment_prompt_v2, temperature=0.7)
print("Improved Sentiment Analysis Result:")
print(result)

Improved Sentiment Analysis Result:
Positive


In [ ]:
product_prompt_v2 = """
Create a product description for a whey protein powder that costs $69.99.

Structure:
- Start with a short, engaging headline.
- Write one paragraph describing the product.
- End with a short call to action.

Length:
- Keep the description between 60 and 100 words.

Style:
- Use a professional, engaging, and customer-friendly tone.
- Make the description suitable for an online store.
- Keep the language clear and concise.

Constraints:
- Only use the provided product information.
- Do not invent ingredients, nutritional values, flavors, or other product specifications.
- Do not mention unverified certifications, testing results, or refund policies.
- Avoid unsupported health claims.
- Keep the price unchanged.
"""

result = call_openai(product_prompt_v2, temperature=0.7)
print("Improved Product Description Result:")
print(result)

Improved Product Description Result:
**Elevate Your Fitness with Premium Whey Protein Powder!**

Unlock your full potential with our high-quality Whey Protein Powder, designed to support your fitness journey and muscle recovery. Each scoop delivers a powerful blend of essential amino acids, perfect for post-workout replenishment or a nutritious addition to your daily routine. With its smooth texture and versatile usage, you can easily mix it into shakes, smoothies, or your favorite recipes. 

Transform your workouts and fuel your body—order your Whey Protein Powder today for just $69.99!


In [6]:
extraction_prompt_v2 = """
Extract information from this customer feedback:
"I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."

Extract the following information:
- Item number
- Order date
- Delivery experience
- Packaging condition

Return the extracted information as a valid JSON object with exactly these keys:
- "item_number"
- "order_date"
- "delivery_experience"
- "packaging_condition"

Only use information provided in the customer feedback.
Do not invent missing information. Use null for missing values.
Do not include explanations, additional text, or Markdown formatting.
"""

result = call_openai(extraction_prompt_v2, temperature=0.7)

print("Improved Data Extraction Result:")
print(result)

Improved Data Extraction Result:
{
  "item_number": "43",
  "order_date": "May 4th",
  "delivery_experience": "took long",
  "packaging_condition": "not damaged"
}


In [ ]:
sentiment_prompt_v3 = """
Classify a customer message.

Choose exactly one of the following classifications:
- Positive
- Negative
- Neutral

Examples:
- "I love this product!" -> Positive
- "This is the worst thing I've ever bought." -> Negative
- "It's okay, nothing special." -> Neutral

Respond with only the selected classification as a single word.
Do not include explanations, additional text, or formatting.

Customer message: "I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."
"""

# Run sentiment_prompt_v3 15 times with temperature=0.7
temperature = 0.7
num_runs = 15

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{sentiment_prompt_v3}'\n")
print("="*80)

sentiment_responses_v3_15 = []

for i in range(num_runs):
    response = call_openai(sentiment_prompt_v3, temperature=temperature)
    sentiment_responses_v3_15.append(response)

    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)

    time.sleep(0.5)  # Small delay to avoid rate limiting

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(sentiment_responses_v3_15))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in sentiment_responses_v3_15) / len(sentiment_responses_v3_15):.0f} characters")

Running the same prompt 15 times with temperature=0.7
Prompt: '
Classify a customer message.

Choose exactly one of the following classifications:
- Positive
- Negative
- Neutral

Examples:
- "I love this product!" -> Positive
- "This is the worst thing I've ever bought." -> Negative
- "It's okay, nothing special." -> Neutral

Respond with only the selected classification as a single word.
Do not include explanations, additional text, or formatting.

Customer message: "I had my initial concerns about the product, but after using it for a week, I am pleasantly surprised by it."
'


🔄 Run #1:
Positive
--------------------------------------------------------------------------------

🔄 Run #2:
Positive
--------------------------------------------------------------------------------

🔄 Run #3:
Positive
--------------------------------------------------------------------------------

🔄 Run #4:
Positive
--------------------------------------------------------------------------------

🔄 Run #5

**Step 9 – Few-Shot Sentiment Analysis Results**

The 15-run test of the few-shot sentiment prompt produced identical results in every run. All responses returned exactly "Positive", resulting in 100% classification accuracy and exact output consistency for the tested message. Compared to version 1, which generated 15 unique responses with varying explanations and formatting, version 3 successfully standardized the output. However, the improvement cannot be attributed to few-shot examples alone, since version 2 had already introduced stricter output constraints.

In [9]:
extraction_prompt_v3 = """
Extract information from this customer feedback:
"I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."

Extract the following information:
- Item number
- Order date
- Delivery experience
- Packaging condition

Return the extracted information as a valid JSON object with exactly these keys:
- "item_number"
- "order_date"
- "delivery_experience"
- "packaging_condition"

Only use information provided in the customer feedback.
Do not invent missing information. Use null for missing values.
Do not include explanations, additional text, or Markdown formatting.

Before generating the final output, follow these steps:
1. Identify the item number and order date.
2. Determine the customer's delivery experience.
3. Identify the packaging condition, paying attention to negations.
4. Verify that each extracted value is supported by the original feedback.

Return only the final JSON object without showing the intermediate analysis.
"""

# Run extraction_prompt_v3 15 times with temperature=0.7
temperature = 0.7
num_runs = 15

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{extraction_prompt_v3}'\n")
print("="*80)

extraction_responses_v3_15 = []

for i in range(num_runs):
    response = call_openai(extraction_prompt_v3, temperature=temperature)
    extraction_responses_v3_15.append(response)

    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)

    time.sleep(0.5)  # Small delay to avoid rate limiting

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(extraction_responses_v3_15))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in extraction_responses_v3_15) / len(extraction_responses_v3_15):.0f} characters")

Running the same prompt 15 times with temperature=0.7
Prompt: '
Extract information from this customer feedback:
"I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."

Extract the following information:
- Item number
- Order date
- Delivery experience
- Packaging condition

Return the extracted information as a valid JSON object with exactly these keys:
- "item_number"
- "order_date"
- "delivery_experience"
- "packaging_condition"

Only use information provided in the customer feedback.
Do not invent missing information. Use null for missing values.
Do not include explanations, additional text, or Markdown formatting.

Before generating the final output, follow these steps:
1. Identify the item number and order date.
2. Determine the customer's delivery experience.
3. Identify the packaging condition, paying attention to negations.
4. Verify that each extracted value is supported by the original feedback.

Return only the final JSON object 

### Step 10 – Chain-of-Thought Data Extraction Results

The 15-run test of the Chain-of-Thought prompt produced 4 unique responses, with an average length of 128 characters. All responses followed the requested JSON structure and included the four required fields. Compared to version 1, which generated 11 unique responses across 15 runs, version 3 showed improved output consistency. However, some inconsistencies remained. The delivery experience was described as either "took long" (53.3%) or "long" (46.7%), although both expressions have the same meaning. More importantly, one response incorrectly added the year 2023 to the order date, while another returned the item number as an integer instead of a string. These results suggest that structured step-by-step instructions can support more consistent extraction, but they do not guarantee fully standardized or accurate outputs. Further constraints may be necessary to prevent unsupported information and inconsistent data types.

In [10]:
product_prompt_v3 = """
Create a product description based on the provided product information.

Structure:
- Start with a short, engaging headline.
- Write one paragraph describing the product.
- End with a short call to action.
- Use the labels "Headline", "Description", and "Call to Action".

Length:
- Keep the description between 60 and 100 words.

Style:
- Use a professional, engaging, and customer-friendly tone.
- Make the description suitable for an online store.
- Keep the language clear and concise.

Constraints:
- Only use the provided product information.
- Do not invent ingredients, nutritional values, flavors, or other product specifications.
- Do not mention unverified certifications, testing results, or refund policies.
- Avoid unsupported health claims.
- Keep the price unchanged.

Examples:

Example 1:
Product: Ceramic coffee mug
Price: $14.99

Headline: Everyday Coffee Moments

Description:
Looking for a ceramic coffee mug? This option keeps things simple: a ceramic mug for coffee, tea, or another favorite drink. It suits familiar moments such as a morning coffee or an afternoon break, without relying on extra features to make its case. Available for $14.99, it offers a straightforward choice for anyone shopping for an everyday mug.

Call to Action:
Make room for your next coffee moment with this ceramic mug.

Example 2:
Product: Paper notebook
Price: $9.99

Headline: A Place for Your Next Idea

Description:
A paper notebook gives you a dedicated place to write down ideas, reminders, and notes. Use it to collect thoughts at home, organize tasks for work, or keep track of things you want to remember. Its purpose is simple and flexible, making it relevant to many everyday routines. Priced at $9.99, this notebook is an option for anyone who prefers to put ideas on paper.

Call to Action:
Keep your next idea close with a paper notebook for $9.99.

Now create a product description for:

Product: Whey protein powder
Price: $69.99

Follow the same structure and style as the examples.
"""

# Run product_prompt_v3 15 times with temperature=0.7
temperature = 0.7
num_runs = 15

print(f"Running the same prompt {num_runs} times with temperature={temperature}")
print(f"Prompt: '{product_prompt_v3}'\n")
print("="*80)

product_responses_v3_15 = []

for i in range(num_runs):
    response = call_openai(product_prompt_v3, temperature=temperature)
    product_responses_v3_15.append(response)

    print(f"\n🔄 Run #{i+1}:")
    print(response)
    print("-"*80)

    time.sleep(0.5)  # Small delay to avoid rate limiting

print("\n" + "="*80)
print("📊 Analysis:")
print(f"Total unique responses: {len(set(product_responses_v3_15))} out of {num_runs}")
print(f"Average length: {sum(len(r) for r in product_responses_v3_15) / len(product_responses_v3_15):.0f} characters")

Running the same prompt 15 times with temperature=0.7
Prompt: '
Create a product description based on the provided product information.

Structure:
- Start with a short, engaging headline.
- Write one paragraph describing the product.
- End with a short call to action.
- Use the labels "Headline", "Description", and "Call to Action".

Length:
- Keep the description between 60 and 100 words.

Style:
- Use a professional, engaging, and customer-friendly tone.
- Make the description suitable for an online store.
- Keep the language clear and concise.

Constraints:
- Only use the provided product information.
- Do not invent ingredients, nutritional values, flavors, or other product specifications.
- Do not mention unverified certifications, testing results, or refund policies.
- Avoid unsupported health claims.
- Keep the price unchanged.

Examples:

Example 1:
Product: Ceramic coffee mug
Price: $14.99

Headline: Everyday Coffee Moments

Description:
Looking for a ceramic coffee mug? This

### Step 11 – Few-Shot Product Description Results

The 15-run test of the few-shot product description prompt showed significant improvements compared to version 1. Although all 15 responses were still unique, the average response length decreased from 2,355 to 590 characters (approximately 75%). All responses followed the required structure with a headline, description, and call to action, while consistently maintaining the correct price of $69.99. The descriptions also became more consistent in style and length. Most importantly, the model no longer generated specific nutritional values, invented certifications, or refund policies. However, some unsupported product claims remained, particularly regarding taste, texture, and mixability. Overall, the combination of few-shot examples, structural requirements, and constraints improved output consistency and reduced hallucinations. Nevertheless, the improvements cannot be attributed to few-shot prompting alone, since version 2 had already introduced several important constraints.

In [11]:
# Step 12 - Sentiment Analysis Variations

sentiment_variations = {
    "Negative": "The product stopped working after two days. I'm very disappointed.",
    "Neutral": "I received my package yesterday and haven't tested it yet.",
    "Mixed": "The product works great, but customer support was terrible."
}

sentiment_variation_results = {}

# Keep the instructions and few-shot examples from v3
sentiment_template = sentiment_prompt_v3.rsplit("Customer message:", 1)[0]

for variation, message in sentiment_variations.items():

    test_prompt = f'{sentiment_template}Customer message: "{message}"'

    response = call_openai(test_prompt, temperature=0.7)

    sentiment_variation_results[variation] = {
        "message": message,
        "response": response
    }

    print(f"\n🔄 Variation: {variation}")
    print(f"Message: {message}")
    print(f"Response: {response}")
    print("-" * 80)


🔄 Variation: Negative
Message: The product stopped working after two days. I'm very disappointed.
Response: Negative
--------------------------------------------------------------------------------

🔄 Variation: Neutral
Message: I received my package yesterday and haven't tested it yet.
Response: Neutral
--------------------------------------------------------------------------------

🔄 Variation: Mixed
Message: The product works great, but customer support was terrible.
Response: Neutral
--------------------------------------------------------------------------------


In [12]:
# Step 12 - Product Description Variations

product_variations = {
    "Electronics": {
        "product": "Wireless headphones",
        "price": "$89.99"
    },
    "Clothing": {
        "product": "Cotton T-shirt",
        "price": "$24.99"
    },
    "Home & Living": {
        "product": "Desk lamp",
        "price": "$39.99"
    }
}

product_variation_results = {}

original_product = "Product: Whey protein powder\nPrice: $69.99"

for variation, details in product_variations.items():

    new_product = f"Product: {details['product']}\nPrice: {details['price']}"

    test_prompt = product_prompt_v3.replace(original_product, new_product)

    response = call_openai(test_prompt, temperature=0.7)

    product_variation_results[variation] = {
        "product": details["product"],
        "price": details["price"],
        "response": response
    }

    print(f"\n🔄 Variation: {variation}")
    print(f"Product: {details['product']}")
    print(f"Price: {details['price']}")
    print(f"\nResponse:\n{response}")
    print("-" * 80)


🔄 Variation: Electronics
Product: Wireless headphones
Price: $89.99

Response:
**Headline:** Unleash Your Sound Experience

**Description:**  
Discover the freedom of sound with our wireless headphones, designed for ultimate comfort and exceptional audio quality. Perfect for music lovers and on-the-go listeners, these headphones provide a seamless connection without the hassle of tangled wires. With a sleek design and lightweight build, they are ideal for extended wear, whether you're at home, commuting, or working out. Available for $89.99, they offer an immersive listening experience that elevates your daily routine.

**Call to Action:**  
Elevate your audio experience today with our wireless headphones for just $89.99!
--------------------------------------------------------------------------------

🔄 Variation: Clothing
Product: Cotton T-shirt
Price: $24.99

Response:
Headline: Comfort Meets Style

Description:  
This cotton T-shirt is the perfect blend of comfort and casual style

In [13]:
# Step 12 - Data Extraction Variations

extraction_variations = {
    "Missing Information": "I ordered item #85. The packaging was damaged.",
    "Negation": "I ordered item #72 on June 12th. The delivery wasn't late and the packaging wasn't damaged.",
    "Complex Feedback": "I ordered item #91 on April 3rd. Delivery was delayed, but the product arrived in perfect condition."
}

extraction_variation_results = {}

original_feedback = "I ordered item #43 on May 4th. The delivery took long but at least the packaging was not damaged."

for variation, feedback in extraction_variations.items():

    test_prompt = extraction_prompt_v3.replace(original_feedback, feedback)

    response = call_openai(test_prompt, temperature=0.7)

    extraction_variation_results[variation] = {
        "feedback": feedback,
        "response": response
    }

    print(f"\n🔄 Variation: {variation}")
    print(f"Feedback: {feedback}")
    print(f"\nResponse:\n{response}")
    print("-" * 80)


🔄 Variation: Missing Information
Feedback: I ordered item #85. The packaging was damaged.

Response:
{
  "item_number": "85",
  "order_date": null,
  "delivery_experience": null,
  "packaging_condition": "damaged"
}
--------------------------------------------------------------------------------

🔄 Variation: Negation
Feedback: I ordered item #72 on June 12th. The delivery wasn't late and the packaging wasn't damaged.

Response:
{
  "item_number": "72",
  "order_date": "June 12th",
  "delivery_experience": "on time",
  "packaging_condition": "not damaged"
}
--------------------------------------------------------------------------------

🔄 Variation: Complex Feedback
Feedback: I ordered item #91 on April 3rd. Delivery was delayed, but the product arrived in perfect condition.

Response:
{
  "item_number": "91",
  "order_date": "April 3rd",
  "delivery_experience": "delayed",
  "packaging_condition": "perfect condition"
}
----------------------------------------------------------------

### Step 12 – Task Variations Results

In Step 12, I tested the improved version 3 prompts with three different input variations for each task to evaluate their robustness. The sentiment analysis prompt correctly classified negative and neutral feedback, while mixed feedback was classified as neutral, highlighting the need for clearer rules when handling conflicting sentiments. The product description prompt maintained a consistent structure, professional style, and correct pricing across all three product categories. However, it still generated unsupported product attributes, such as adjustable brightness for a desk lamp. The data extraction prompt successfully handled missing information and negations but incorrectly interpreted the condition of a product as the condition of its packaging in the complex feedback example. Overall, the tests showed that the improved prompts perform well in terms of formatting and basic task execution, but some semantic errors and unsupported claims remain. Further adjustments could improve reliability across different inputs. Since each variation was tested only once, the results provide initial observations rather than definitive consistency measurements.

In [ ]:
# Use the original documented results from the 15-run tests
comparison_data = [
    {"Task": "Sentiment", "Version": "v1", "Runs": 15,
     "Unique Responses": 15, "Avg. Length": 174},

    {"Task": "Sentiment", "Version": "v3", "Runs": 15,
     "Unique Responses": 1, "Avg. Length": 8},

    {"Task": "Product", "Version": "v1", "Runs": 15,
     "Unique Responses": 15, "Avg. Length": 2355},

    {"Task": "Product", "Version": "v3", "Runs": 15,
     "Unique Responses": 15, "Avg. Length": 590},

    {"Task": "Extraction", "Version": "v1", "Runs": 15,
     "Unique Responses": 11, "Avg. Length": 194},

    {"Task": "Extraction", "Version": "v3", "Runs": 15,
     "Unique Responses": 4, "Avg. Length": 128}
]

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Calculate percentage of unique responses
comparison_df["Unique Rate (%)"] = (
    comparison_df["Unique Responses"]
    / comparison_df["Runs"] * 100
).round(1)

display(comparison_df)

,Task,Version,Runs,Unique Responses,Avg. Length,Unique Rate (%)
0,Sentiment,v1,15,15,174,100.0
1,Sentiment,v3,15,1,8,6.7
2,Product,v1,15,15,2355,100.0
3,Product,v3,15,15,590,100.0
4,Extraction,v1,15,11,194,73.3
5,Extraction,v3,15,4,128,26.7


In [17]:
# Documented improvement metrics from 15-run tests
improvement_data = [
    {"Task": "Sentiment", "Metric": "Correct Classification",
     "v1": 100.0, "v3": 100.0},

    {"Task": "Sentiment", "Metric": "Single-Word Output",
     "v1": 0.0, "v3": 100.0},

    {"Task": "Product", "Metric": "Correct Price",
     "v1": 100.0, "v3": 100.0},

    {"Task": "Product", "Metric": "No Unsupported Numerical Nutritional Values",
     "v1": 0.0, "v3": 100.0},

    {"Task": "Extraction", "Metric": "Valid JSON",
     "v1": 0.0, "v3": 100.0},

    {"Task": "Extraction", "Metric": "Fully Correct Extraction",
     "v1": 93.3, "v3": 93.3}
]

improvement_df = pd.DataFrame(improvement_data)

# Calculate improvement in percentage points
improvement_df["Improvement (pp)"] = (
    improvement_df["v3"] - improvement_df["v1"]
).round(1)

display(improvement_df)

,Task,Metric,v1,v3,Improvement (pp)
0,Sentiment,Correct Classification,100.0,100.0,0.0
1,Sentiment,Single-Word Output,0.0,100.0,100.0
2,Product,Correct Price,100.0,100.0,0.0
3,Product,No Unsupported Numerical Nutritional Values,0.0,100.0,100.0
4,Extraction,Valid JSON,0.0,100.0,100.0
5,Extraction,Fully Correct Extraction,93.3,93.3,0.0


## Step 13 – Final Evaluation and Comparison Report

### 1. Overview

The objective of this lab was to improve the consistency and reliability of three different AI tasks: Sentiment Analysis, Product Description Generation, and Data Extraction.

I started with simple zero-shot prompts (v1), identified failure patterns through repeated testing, and improved the prompts by introducing clearer instructions, output constraints, structured formats, few-shot examples, and step-by-step reasoning instructions.

All final version 3 prompts were tested 15 times using GPT-4o-mini with a temperature of 0.7 to compare their performance against the initial versions.

### 2. Quantitative Comparison

| Metric | Version 1 | Version 3 |
|---|---|---|
| Sentiment – Unique Responses | 15/15 | 1/15 |
| Sentiment – Correct Classification | 100% | 100% |
| Sentiment – Single-Word Output | 0% | 100% |
| Product – Average Response Length | 2,355 characters | 590 characters |
| Product – Correct Price | 100% | 100% |
| Product – No Unsupported Numerical Nutritional Values | 0% | 100% |
| Extraction – Unique Responses | 11/15 | 4/15 |
| Extraction – Valid JSON | 0% | 100% |
| Extraction – Fully Correct Extraction | 93.3% | 93.3% |

The results show improvements in output standardization and the prevention of specific unsupported claims. However, a lower number of unique responses does not automatically indicate better factual accuracy, especially for creative tasks.

### 3. Key Findings by Task

**Sentiment Analysis**

The sentiment analysis prompt already achieved 100% correct classification for the original customer message in version 1. However, the responses varied in wording, formatting, and length.

By introducing explicit classification categories, single-word output constraints, and few-shot examples, version 3 generated exactly the same response ("Positive") in all 15 runs.

This demonstrates a significant improvement in output consistency. However, the task variation tests also showed that mixed customer feedback can be ambiguous and may require additional classification rules.

**Product Description Generation**

The product description prompt showed the greatest improvement in controlling response length and structure. The average response length decreased by approximately 75%, from 2,355 to 590 characters.

All version 3 responses followed the required structure and maintained the correct product price. Unlike version 1, none of the 15 final responses included specific invented nutritional values, certifications, or refund policies.

However, some unsupported product attributes remained, including claims about taste, texture, and mixability. The task variation tests revealed similar problems with other products, such as invented technical features for wireless headphones and desk lamps.

These results suggest that few-shot examples and clear constraints improve consistency but cannot completely prevent unsupported claims.

**Data Extraction**

The data extraction prompt improved significantly in output structure. While version 1 generated differently formatted lists with inconsistent field names, version 3 produced valid JSON in all 15 runs.

The number of unique responses decreased from 11 to 4, indicating greater standardization.

However, extraction accuracy did not improve according to our selected evaluation criteria, remaining at 93.3%. Version 3 incorrectly added an unspecified year in one response and returned an inconsistent data type in another.

The task variation tests also revealed that the model sometimes confused the condition of a product with the condition of its packaging.

This shows that structured output and step-by-step instructions improve formatting reliability but do not guarantee semantic accuracy.

### 4. Effectiveness of Prompt Engineering Techniques

The different techniques addressed different weaknesses:

- **Clear instructions and constraints** helped standardize classification outputs and reduce unsupported product details.
- **Few-shot prompting** demonstrated the expected structure and format, particularly for sentiment analysis and product descriptions.
- **Structured JSON output** improved consistency and made extracted information easier to process programmatically.
- **Step-by-step reasoning instructions** provided a more systematic extraction approach, although some semantic errors remained.

Since multiple improvements were introduced between versions 1 and 3, their individual contributions cannot be determined precisely. Version 2 was also tested only once per task, so the available results support a quantitative comparison between versions 1 and 3, but only a qualitative assessment of version 2.

### 5. Limitations and Future Improvements

Although the final prompts performed better in several areas, the evaluation has some limitations.

The repeated 15-run tests used only one fixed input per task. Additional task variations were tested only once each, making it difficult to evaluate general reliability across different scenarios.

Furthermore, some evaluation metrics relied on simple pattern matching, which cannot identify every possible semantic error or unsupported statement.

In future iterations, I would test a larger variety of inputs, introduce stricter validation rules for structured outputs, and provide verified product information to reduce unsupported claims. I would also test the effects of individual prompt changes separately to better understand which techniques contribute most to improved performance.

### 6. Conclusion

Overall, the lab demonstrated that systematic prompt engineering can significantly improve the consistency and reliability of AI-generated responses.

The strongest improvements were achieved in output formatting, response length, and the reduction of unsupported product details. However, the results also showed that standardized outputs do not necessarily guarantee factual accuracy.

The most important lesson was that prompt quality should be evaluated through repeated testing, clearly defined metrics, and task-specific failure analysis rather than relying on a single successful response.